# Phân tích SQL – Dự đoán phí vận chuyển đơn hàng thương mại điện tử Olist
**Nhóm 8 · Lớp AI2107 · Môn ADY201M · FALL26**

---
## 1. Mục tiêu của bước SQL

Sau bước làm sạch, nhóm dùng SQL để **khám phá bảng dữ liệu sạch trước khi xây mô hình**. Bước này trả lời các câu hỏi nghiên cứu sau:

- **RQ1.** Đặc điểm vật lý của sản phẩm (cân nặng, thể tích, giá trị) có ảnh hưởng như thế nào đến phí vận chuyển (`freight_value`)?
- **RQ2.** Khoảng cách địa lý giữa người bán và người mua có mối quan hệ như thế nào với phí vận chuyển?
- **RQ3.** Số lượng sản phẩm trong đơn hàng và thời điểm đặt hàng có mối quan hệ như thế nào với phí vận chuyển?
- **RQ4.** Có thể sử dụng các đặc điểm về sản phẩm, khoảng cách và đơn hàng để xây dựng mô hình dự đoán phí vận chuyển hay không?





## 2. Dữ liệu sử dụng

- **Nguồn:** Brazilian E-Commerce Public Dataset by Olist (Kaggle), giấy phép CC BY-NC-SA 4.0.
- **Bảng dùng trong SQL:** `olist_freight_clean` (108.937 dòng), là kết quả của bước làm sạch: chỉ giữ đơn đã giao, giai đoạn 01/2017–08/2018, bỏ dòng thiếu tọa độ, cân nặng hoặc kích thước, bỏ phí vận chuyển bằng 0.
- **Đơn vị quan sát:** một sản phẩm trong một đơn hàng, xác định bởi cặp (`order_id`, `order_item_id`).

| Cột | Ý nghĩa |
|---|---|
| `freight_value` | Phí vận chuyển (BRL), **biến cần dự đoán** |
| `price` | Giá bán của sản phẩm (BRL) |
| `weight_g` | Cân nặng sản phẩm (gam) |
| `length_cm`, `height_cm`, `width_cm`, `volume_cm3` | Kích thước và thể tích sản phẩm |
| `category` | Danh mục sản phẩm (tiếng Anh; `unknown` nếu thiếu) |
| `seller_state`, `customer_state` | Bang của người bán và của khách |
| `same_state` | 1 nếu người bán và khách cùng bang, 0 nếu khác bang |
| `distance_km` | Khoảng cách giữa người bán và khách (km), tính theo công thức Haversine từ tọa độ mã bưu chính |
| `order_purchase_timestamp`, `month`, `t_days` | Thời điểm đặt hàng, tháng đặt hàng, số ngày tính từ 01/01/2017 |
| `order_n_items` | Số sản phẩm trong đơn |

## 3. Chuẩn bị môi trường
Nạp file dữ liệu sạch vào SQLite (cơ sở dữ liệu chạy trong bộ nhớ) và khai báo hàm `sql()` để chạy câu lệnh.

In [ ]:
import sqlite3, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/olist_freight_clean_v3.csv'   # <-- sửa đúng đường dẫn
df = pd.read_csv(CSV_PATH)

conn = sqlite3.connect(':memory:')
df.to_sql('olist_freight_clean', conn, index=False, if_exists='replace')

def sql(query):
    """Chạy một câu SQL và trả về bảng kết quả"""
    return pd.read_sql_query(query, conn)

print(df.shape)

### Kiểm tra dữ liệu đã nạp
Xác nhận bảng nạp vào đúng với kết quả của bước làm sạch.

In [ ]:
print('Số dòng, số cột        :', df.shape)
print('Tổng ô thiếu           :', df.isna().sum().sum())
print('Số dòng trùng khóa      :', df.duplicated(['order_id', 'order_item_id']).sum())
print('Giai đoạn đặt hàng      :', df['month'].min(), 'đến', df['month'].max())

Số dòng, số cột        : (108937, 27)
Tổng ô thiếu           : 0
Số dòng trùng khóa      : 0
Giai đoạn đặt hàng      : 2017-01 đến 2018-08


## 4. Các truy vấn

Tám truy vấn (Query 1–8), sắp xếp từ tổng quan, đến các yếu tố ảnh hưởng đến phí, đến kiểm tra giá trị bất thường.

---
## Query 1 — Tổng quan dữ liệu và phí vận chuyển

**Câu hỏi phân tích**: Dữ liệu có bao nhiêu dòng, bao nhiêu đơn hàng? Phí vận chuyển nhỏ nhất, lớn nhất và trung bình là bao nhiêu?
**Mục đích**: Nắm quy mô dữ liệu và đặc điểm của biến mục tiêu `freight_value` trước khi đi vào từng yếu tố ảnh hưởng.

In [ ]:
print("=== 1. TỔNG QUAN DỮ LIỆU VÀ PHÍ VẬN CHUYỂN ===")
print(sql("""
SELECT COUNT(*) AS so_dong,
       COUNT(DISTINCT order_id) AS so_don_hang,
       MIN(freight_value) AS phi_nho_nhat,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB,
       MAX(freight_value) AS phi_lon_nhat
FROM olist_freight_clean;
"""))


=== 1. TỔNG QUAN DỮ LIỆU VÀ PHÍ VẬN CHUYỂN ===
   so_dong  so_don_hang  phi_nho_nhat  phi_ship_TB  phi_lon_nhat
0   108937        95377          0.01        20.01        409.68


**Nhận xét**: Dữ liệu có **108.937** dòng thuộc **95.377** đơn hàng. Phí trung bình là 20,01 R\$ nhưng phí cao nhất lên tới 409,68 R\$ (gấp khoảng **20 lần** mức trung bình) — phí lệch phải mạnh, cần để ý các giá trị lớn khi huấn luyện mô hình và cân nhắc log-transform.

---
## Query 2 — Phí ship trung bình theo nhóm trọng lượng

**Câu hỏi phân tích**: Hàng càng nặng thì phí vận chuyển có tăng không?
**Mục đích**: Đo trực tiếp mối quan hệ trọng lượng–phí ship bằng cách chia nhóm, bổ sung bằng chứng SQL độc lập cho bước Regression Analysis.

In [ ]:
print("=== 2. PHÍ SHIP THEO NHÓM TRỌNG LƯỢNG ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN weight_g < 500 THEN '1. Rất nhẹ (<500g)'
               WHEN weight_g < 2000 THEN '2. Nhẹ (500-2000g)'
               WHEN weight_g < 5000 THEN '3. Trung bình (2-5kg)'
               ELSE '4. Nặng (>5kg)' END AS nhom_trong_luong
    FROM olist_freight_clean
)
SELECT nhom_trong_luong,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_trong_luong
ORDER BY nhom_trong_luong;
"""))


=== 2. PHÍ SHIP THEO NHÓM TRỌNG LƯỢNG ===
        nhom_trong_luong  so_dong  phi_ship_TB
0     1. Rất nhẹ (<500g)    43532        15.17
1     2. Nhẹ (500-2000g)    40632        17.89
2  3. Trung bình (2-5kg)    11220        22.81
3         4. Nặng (>5kg)    13553        39.62


**Nhận xét**: Phí ship tăng từ 15,17 R\$ (rất nhẹ) lên 39,62 R\$ (nặng) — tăng **2,6 lần**. Trọng lượng là đầu vào có ảnh hưởng rõ, phí tăng dần theo từng nhóm.

---
## Query 3 — Phí ship trung bình theo nhóm khoảng cách

**Câu hỏi phân tích**: Khoảng cách giao hàng (người bán → người mua) ảnh hưởng thế nào đến phí vận chuyển?
**Mục đích**: Kiểm chứng giả thuyết "khoảng cách càng xa, phí ship càng cao" — một trong những biến đầu vào (X) chính của mô hình Regression.

In [ ]:
print("=== 3. PHÍ SHIP THEO NHÓM KHOẢNG CÁCH ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN distance_km < 200 THEN '1. Gần (<200km)'
               WHEN distance_km < 800 THEN '2. Trung bình (200-800km)'
               WHEN distance_km < 1500 THEN '3. Xa (800-1500km)'
               ELSE '4. Rất xa (>1500km)' END AS nhom_khoang_cach
    FROM olist_freight_clean
)
SELECT nhom_khoang_cach,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom_khoang_cach
ORDER BY nhom_khoang_cach;
"""))


=== 3. PHÍ SHIP THEO NHÓM KHOẢNG CÁCH ===
            nhom_khoang_cach  so_dong  phi_ship_TB
0            1. Gần (<200km)    28096        12.66
1  2. Trung bình (200-800km)    54032        19.80
2         3. Xa (800-1500km)    17099        23.74
3        4. Rất xa (>1500km)     9710        35.91


**Nhận xét**: Phí ship tăng đều theo khoảng cách — đơn "rất xa" (>1500km) có phí trung bình 35,91 R\$, cao gấp **~2,8 lần** đơn "gần" (<200km, 12,66 R\$). Khoảng cách là yếu tố ảnh hưởng rõ.

---
## Query 4 — So sánh phí ship: cùng bang vs khác bang

**Câu hỏi phân tích**: Giao hàng trong cùng 1 bang có rẻ hơn đáng kể so với giao khác bang không?
**Mục đích**: Đánh giá vai trò của biến phân loại địa lý (`same_state`) bên cạnh biến khoảng cách liên tục.

In [ ]:
print("=== 4. SO SÁNH PHÍ SHIP: CÙNG BANG VS KHÁC BANG ===")
print(sql("""
SELECT CASE WHEN same_state = 1 THEN 'Cùng bang'
            ELSE 'Khác bang'
       END AS nhom,
       COUNT(*) AS so_dong,
       ROUND(AVG(distance_km), 0) AS khoang_cach_TB,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM olist_freight_clean
GROUP BY same_state;
"""))


=== 4. SO SÁNH PHÍ SHIP: CÙNG BANG VS KHÁC BANG ===
        nhom  so_dong  khoang_cach_TB  phi_ship_TB
0  Khác bang    69330           848.0        23.74
1  Cùng bang    39607           154.0        13.48


**Nhận xét**: Đơn khác bang có phí ship trung bình cao hơn **76%** so với cùng bang (23,74 R\$ so với 13,48 R\$), phần lớn do khoảng cách xa hơn (848km so với 154km). `same_state` liên quan chặt với khoảng cách nên khi giải thích mô hình cần nói rõ hai biến này cùng phản ánh một ý.

---
## Query 5 — Top 10 danh mục sản phẩm có phí ship trung bình cao nhất

**Câu hỏi phân tích**: Danh mục sản phẩm nào có chi phí vận chuyển cao nhất, và có liên hệ gì với trọng lượng?
**Mục đích**: Kiểm tra xem phí ship cao có thực sự gắn liền với các mặt hàng nặng/cồng kềnh hay không (chỉ xét danh mục có ≥300 dòng để đảm bảo độ tin cậy thống kê).

In [ ]:
print("=== 5. TOP 10 DANH MỤC PHÍ SHIP CAO NHẤT ===")
print(sql("""
SELECT category,
       COUNT(*) AS so_dong,
       ROUND(AVG(weight_g), 0) AS trong_luong_TB,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM olist_freight_clean
GROUP BY category
HAVING COUNT(*) >= 300
ORDER BY phi_ship_TB DESC
LIMIT 10;
"""))


=== 5. TOP 10 DANH MỤC PHÍ SHIP CAO NHẤT ===
                          category  so_dong  trong_luong_TB  phi_ship_TB
0                 office_furniture     1655         11329.0        40.17
1            furniture_living_room      443          8727.0        37.43
2              luggage_accessories     1073          5788.0        27.83
3              musical_instruments      650          3135.0        27.46
4        construction_tools_lights      300          3358.0        24.80
5                 small_appliances      658          2929.0        23.33
6                     garden_tools     4196          2827.0        22.94
7                home_construction      596          3037.0        22.84
8                             baby     2961          3271.0        22.24
9  construction_tools_construction      909          3080.0        22.08


**Nhận xét**: Đắt nhất là đồ nội thất văn phòng (40,17 R\$) và nội thất phòng khách (37,43 R\$), đồng thời cũng là hai nhóm nặng nhất (khoảng 11,3kg và 8,7kg). Danh mục đắt phần lớn vì hàng nặng chứ không phải vì bản thân danh mục — cần giữ trọng lượng trong mô hình để không hiểu sai vai trò của danh mục.

---
## Query 6 — Phí ship theo số lượng sản phẩm trong đơn hàng

**Câu hỏi phân tích**: Đơn hàng mua nhiều sản phẩm cùng lúc có phí ship trung bình trên mỗi sản phẩm thấp hơn không (hiệu ứng gộp đơn)?
**Mục đích**: Kiểm tra đầu vào `order_n_items` (số sản phẩm trong đơn) và xem việc gộp nhiều sản phẩm vào 1 đơn có ảnh hưởng đến phí mỗi dòng hay không.

In [ ]:
print("=== 6. PHÍ SHIP THEO SỐ LƯỢNG SẢN PHẨM TRONG ĐƠN (order_n_items) ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN order_n_items = 1 THEN '1 sản phẩm'
               ELSE '>=2 sản phẩm' END AS nhom
    FROM olist_freight_clean
)
SELECT nhom,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom;
"""))


=== 6. PHÍ SHIP THEO SỐ LƯỢNG SẢN PHẨM TRONG ĐƠN (order_n_items) ===
           nhom  so_dong  phi_ship_TB
0    1 sản phẩm    85856        20.39
1  >=2 sản phẩm    23081        18.59


**Nhận xét**: Đơn có từ 2 sản phẩm trở lên có phí ship trung bình mỗi dòng thấp hơn một chút (18,59 R\$ so với 20,39 R\$, thấp hơn khoảng 1,8 R\$) — gợi ý phí được chia hoặc tính chung theo cả đơn, cần kiểm tra thêm ở bước phân tích Python.

---
## Query 7 — Phí ship trung bình theo tháng (xu hướng mùa vụ)

**Câu hỏi phân tích**: Phí vận chuyển có biến động theo thời gian/mùa vụ trong năm không?
**Mục đích**: Kiểm tra yếu tố thời gian — nếu có biến động mùa vụ rõ rệt (VD: cao điểm mua sắm cuối năm), có thể cân nhắc thêm biến tháng/mùa vào mô hình Regression.

In [ ]:
print("=== 7. PHÍ SHIP TRUNG BÌNH THEO THÁNG (XU HƯỚNG MÙA VỤ) ===")
print(sql("""
SELECT month,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM olist_freight_clean
GROUP BY month
ORDER BY month;
"""))


=== 7. PHÍ SHIP TRUNG BÌNH THEO THÁNG (XU HƯỚNG MÙA VỤ) ===
      month  so_dong  phi_ship_TB
0   2017-01      912        17.17
1   2017-02     1858        19.92
2   2017-03     2883        19.00
3   2017-04     2548        19.53
4   2017-05     3983        19.34
5   2017-06     3459        19.53
6   2017-07     4393        19.15
7   2017-08     4756        19.03
8   2017-09     4711        19.77
9   2017-10     5178        19.74
10  2017-11     8435        19.52
11  2017-12     6152        18.92
12  2018-01     7998        19.07
13  2018-02     7478        18.58
14  2018-03     7981        20.86
15  2018-04     7634        20.74
16  2018-05     7558        19.91
17  2018-06     6981        22.23
18  2018-07     6934        22.95
19  2018-08     7105        20.57


**Nhận xét**: Phần lớn các tháng phí dao động quanh 19–20 R\$; **không thấy cao điểm cuối năm** (tháng 11 và 12/2017: 19,52 và 18,92 R\$). Tháng 6 và 7/2018 cao hơn (22,24 và 22,96 R\$) nhưng không có xu hướng tăng đều. Yếu tố thời gian ảnh hưởng nhẹ hơn nhiều so với khoảng cách và trọng lượng; dữ liệu chỉ có 20 tháng (các tháng 1–8 xuất hiện 2 lần, tháng 9–12 chỉ 1 lần) nên chưa đủ để kết luận chắc về mùa vụ. Nên chia dữ liệu theo thời gian (các tháng cuối để kiểm tra) khi huấn luyện.

---
## Query 8 — Top 10 đơn hàng có phí ship cao nhất

**Câu hỏi phân tích**: Những đơn có phí cao bất thường là đơn như thế nào? Có phải lỗi dữ liệu không?
**Mục đích**: Kiểm tra giá trị ngoại lai (outlier) của biến mục tiêu để quyết định giữ lại hay loại bỏ trước khi huấn luyện.

In [ ]:
print("=== 8. TOP 10 ĐƠN HÀNG PHÍ SHIP CAO NHẤT ===")
print(sql("""
SELECT order_id, category, weight_g, ROUND(distance_km, 0) AS khoang_cach_km, freight_value
FROM olist_freight_clean
ORDER BY freight_value DESC
LIMIT 10;
"""))


=== 8. TOP 10 ĐƠN HÀNG PHÍ SHIP CAO NHẤT ===
                           order_id  ... freight_value
0  a77e1550db865202c56b19ddc6dc4d53  ...        409.68
1  076d1555fb53a89b0ef4d529e527a0f6  ...        375.28
2  3fde74c28a3d5d618c00f26d51baafa0  ...        375.28
3  9f49bd16053df810384e793386312674  ...        339.59
4  264a7e199467906c0727394df82d1a6a  ...        338.30
5  c7a07ddd52bbe18b61da49a8d89853d3  ...        322.10
6  0b6230647ed16f4b3e70282dc4b5b87f  ...        321.88
7  0822bcde10bb5d023755a71bc8f7797f  ...        321.46
8  43bdbd9dc0931d72befdf4765af6c442  ...        317.47
9  6ddfbf514959b49b6410c01ad93054bb  ...        314.40

[10 rows x 5 columns]


**Nhận xét**: Cả 10 đơn đều là hàng nặng từ 11,3kg đến 30kg, phí từ 314 đến 410 R\$ — phí cao do hàng nặng thật, không phải lỗi dữ liệu, nên giữ lại các dòng này.

---
## 5. Tổng hợp phát hiện

| Query | Phát hiện | Quyết định cho bước sau |
|---|---|---|
| 1 | Phí trung bình 20,01 R\$, cao nhất 409,68 R\$ (gấp khoảng 20 lần) | Phí lệch phải, cân nhắc log-transform khi mô hình |
| 2 | Phí tăng từ 15,17 lên 39,62 R\$ theo trọng lượng | Giữ `weight_g` làm đầu vào |
| 3 | Phí tăng từ 12,66 lên 35,91 R\$ theo khoảng cách | Giữ `distance_km` làm đầu vào |
| 4 | Cùng bang 13,48 R\$, khác bang 23,74 R\$ | Giữ `same_state`, lưu ý liên quan chặt với khoảng cách |
| 5 | Nội thất văn phòng và phòng khách đắt nhất, cũng là nhóm nặng nhất | Giữ trọng lượng cùng danh mục để tránh hiểu sai vai trò danh mục |
| 6 | Đơn từ 2 sản phẩm có phí mỗi dòng thấp hơn (18,59 so với 20,39 R\$) | Cân nhắc đơn vị dự đoán là từng sản phẩm hay cả đơn; kiểm tra thêm ở Python |
| 7 | Phí quanh 19–20 R\$ phần lớn các tháng, cao hơn vào tháng 6–7/2018, không có cao điểm cuối năm | Chia dữ liệu huấn luyện và kiểm tra theo thời gian |
| 8 | 10 đơn phí cao nhất đều là hàng nặng 11,3–30 kg | Giữ lại, không phải lỗi dữ liệu |

## 6. Hạn chế và bước tiếp theo

**Hạn chế:**
- Phí vận chuyển có thể là phí của cả đơn được chia hoặc lặp cho từng sản phẩm (gợi ý từ Query 6), nên phí theo từng dòng chưa chắc phản ánh đúng chi phí của riêng sản phẩm đó; cần kiểm tra thêm ở bước Python.
- Khoảng cách tính từ tọa độ mã bưu chính, không phải địa chỉ chi tiết.
- Dữ liệu không có thông tin hãng vận chuyển, giá cước hay giá nhiên liệu.
- Kết quả SQL chỉ mô tả mối quan hệ, chưa phải kết luận về nguyên nhân.

**Bước tiếp theo (Python):** vẽ biểu đồ phân phối và quan hệ giữa phí với từng biến, tính tương quan (gồm cặp trọng lượng – thể tích), rồi huấn luyện và so sánh các mô hình hồi quy bằng R², MAE và cross-validation.

In [ ]:
conn.close()
print('\nĐã đóng kết nối SQLite.')
